<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<span style="display:inline-block;margin-bottom:10px;padding:2px 12px;border-radius:999px;background:#6366f1;color:#fff;font:600 11px/20px system-ui,sans-serif;letter-spacing:.05em;text-transform:uppercase">Notebook 08</span>
<h1 style="font-size:1.35em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">The production backend: check every new part</h1>
<p style="margin:.55em 0">Notebooks 01–07 grew the voice desk in <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">live/</code>. <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">app/</code> is the same system rebuilt to run in
production, organised so that <b>adding a function is the only step</b>:</p>
<div style="display:flex;flex-direction:column;align-items:flex-start;gap:4px;margin:.8em 0">
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>app/skills/*.py</b>: typed functions with docstrings. Discovered at startup; schema from type hints.</div>
  <div style="opacity:.6;padding-left:18px">▼</div>
  <div style="border:2px solid #6366f1;background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>registry</b>: validate → guard (<code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">needs_patient</code>) → run with a timeout · errors become data</div>
  <div style="opacity:.6;padding-left:18px">▼</div>
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>Session</b>: one conversation's state · the agent's allowlist · its prompt from <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">prompts/*.md</code></div>
  <div style="opacity:.6;padding-left:18px">▼</div>
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>realtime</b>: OpenAI Realtime, tool calls, barge-in, filler, stall recovery (only <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">websockets</code>)</div>
  <div style="opacity:.6;padding-left:18px">▼</div>
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>server</b>: /ws?agent= · /healthz · /metrics · limits · JSON logs · graceful drain</div>
</div>
<p style="margin:.55em 0">Everything tunable is in <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">config/*.toml</code> and <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">prompts/*.md</code>, overridable by environment variables.</p>

</div>

In [1]:
import asyncio, json, os, re, sys, time
from pathlib import Path

from app import registry, selfcheck, server, settings, store
from app.metrics import METRICS
from app.session import Session

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">1. Settings: one file, environment overrides, typos fail</h2>
<p style="margin:.55em 0">A wrong key or type stops the server at startup instead of being silently ignored:</p>

</div>

In [2]:
s = settings.load()
s.openai.model, s.agents['front_desk'].toolsets, s.agents['front_desk'].vad

('gpt-realtime-mini',
 ('identity', 'registration', 'appointments', 'clinic_info'),
 {'type': 'server_vad', 'silence_duration_ms': 900})

In [3]:
s2 = settings.load(env={'APP__OPENAI__VOICE': 'cedar', 'APP__SERVER__MAX_SESSIONS': '5'})
s2.openai.voice, s2.server.max_sessions

('cedar', 5)

In [4]:
try:
    settings.load(env={'APP__SERVER__PROT': '9000'})
except settings.SettingsError as e:
    print('refused:', e)

refused: server: unknown key(s) ['prot']


In [5]:
selfcheck.check_settings()

settings ok


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">2. Skills: discovered, described by their own code</h2>
<p style="margin:.55em 0"><code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">discover()</code> imports every module in <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">app/skills/</code>. The file name is the toolset:</p>

</div>

In [6]:
registry.discover()

{'appointments': ['find_slots',
  'book_appointment',
  'my_appointments',
  'cancel_appointment'],
 'clinic_info': ['list_doctors', 'clinic_hours'],
 'identity': ['verify_patient', 'confirm_code', 'choose_patient'],
 'orders': ['get_order_status', 'convert_temperature'],
 'registration': ['start_registration']}

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">The tool definition the model receives is generated from the function. Here is one with an enum and optional arguments:</p>

</div>

In [7]:
registry.REGISTRY['start_registration'].spec

{'type': 'function',
 'name': 'start_registration',
 'description': "Validate a NEW patient's details and text a 6-digit code to their phone. Call again with corrections.",
 'parameters': {'type': 'object',
  'properties': {'first_name': {'type': 'string'},
   'last_name': {'type': 'string'},
   'phone': {'type': 'string',
    'description': '10-digit mobile number, digits only'},
   'sex': {'type': 'string', 'enum': ['Male', 'Female']},
   'dob': {'type': 'string',
    'description': 'YYYY-MM-DD, or empty if only age is known'},
   'age': {'type': 'string',
    'description': 'age in years, or empty if dob is given'},
   'email': {'type': 'string', 'description': 'empty if not given'}},
  'required': ['first_name', 'last_name', 'phone', 'sex'],
  'additionalProperties': False}}

In [8]:
await selfcheck.check_registry()

skill s timed out after 0.2s


skill b failed
Traceback (most recent call last):
  File "D:\Code\realtime-talk\app\registry.py", line 175, in call
    out = await asyncio.wait_for(asyncio.to_thread(s.fn, ctx, **args), timeout)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\aksha\AppData\Local\Programs\Python\Python311\Lib\asyncio\tasks.py", line 479, in wait_for
    return fut.result()
           ^^^^^^^^^^^^
  File "C:\Users\aksha\AppData\Local\Programs\Python\Python311\Lib\asyncio\threads.py", line 25, in to_thread
    return await loop.run_in_executor(None, func_call)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\aksha\AppData\Local\Programs\Python\Python311\Lib\concurrent\futures\thread.py", line 58, in run
    result = self.fn(*self.args, **self.kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\Code\realtime-talk\app\selfcheck.py", line 122, in <lambda>
    boom = registry.Skill("b", "t", "d", "", guarded.parameters, la

skill n returned list, not a dict


registry ok


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">3. Adding a function: nothing else changes</h2>
<p style="margin:.55em 0">A brand-new module, written to a separate package so <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">app/</code> stays untouched. Discovery finds it, an
agent lists its toolset (here through an environment variable), and it appears in the agent's tools <b>and</b> prompt.</p>

</div>

In [9]:
Path('demo_skills').mkdir(exist_ok=True)
Path('demo_skills/__init__.py').write_text('')
Path('demo_skills/pharmacy.py').write_text('''
from app.registry import skill

@skill
def pharmacy_hours(ctx, day: str = "today") -> dict:
    """When the clinic pharmacy is open.

    When: when asked about the pharmacy.
    Args:
        day: a weekday name, or today
    """
    return {"status": "ok", "hours": "8 AM to 8 PM every day"}
''')
registry.discover('demo_skills')

{'appointments': ['find_slots',
  'book_appointment',
  'my_appointments',
  'cancel_appointment'],
 'clinic_info': ['list_doctors', 'clinic_hours'],
 'identity': ['verify_patient', 'confirm_code', 'choose_patient'],
 'orders': ['get_order_status', 'convert_temperature'],
 'registration': ['start_registration'],
 'pharmacy': ['pharmacy_hours']}

In [10]:
s3 = settings.load(env={'APP__AGENTS__FRONT_DESK__TOOLSETS':
                        json.dumps([*s.agents['front_desk'].toolsets, 'pharmacy'])})
demo = Session(s3, 'front_desk')
'pharmacy_hours' in demo.tools, [l for l in demo.instructions().splitlines() if 'pharmacy' in l]

(True, ['- pharmacy_hours: when asked about the pharmacy.'])

In [11]:
await demo.run_tool('pharmacy_hours', '{}')

{'status': 'ok', 'hours': '8 AM to 8 PM every day'}

In [12]:
import shutil
shutil.rmtree('demo_skills')
registry.REGISTRY.pop('pharmacy_hours'); sorted(registry.discover())

['appointments', 'clinic_info', 'identity', 'orders', 'registration']

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">4. The session: state and guards, in code</h2>
<p style="margin:.55em 0">The whole front-desk flow through <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">Session.run_tool</code>, the one door from the model to our code:
guard, wrong code, shared phone, someone else's appointment, booking, registration, and the SMS-pumping cap.</p>

</div>

In [13]:
await selfcheck.check_session()

otp send limit reached (3)


session ok


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">5. Stall recovery</h2>
<p style="margin:.55em 0">A fake OpenAI goes silent mid-reply and stops answering pings, like the real stalls measured in notebook 07.
The conversation reconnects, replays what was said, and the reply completes.</p>

</div>

In [14]:
await selfcheck.check_resume()

RESUME Stalled: reconnecting (1/2), replaying 1 items


resume ok: stall detected, reconnected, reply 4.0s after the start


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">6. The server, end to end, against a scripted fake OpenAI</h2>
<p style="margin:.55em 0">The real server (<code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">server.run</code>) talks to a local websocket that speaks the Realtime protocol from a
script: it "hears" a phone number and proposes <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">verify_patient</code>. Checked: page, health, 404, unknown
agent refused, the tool call runs, the SMS reaches the browser, <b>the code never reaches the model</b>, the filler
plays, metrics count it, and a second caller gets "busy" when the limit is one.</p>

</div>

In [15]:
await selfcheck.check_server()

server ok: page, health, metrics, agent allowlist, tool call + SMS end to end, busy limit


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">7. One real conversation through the production server</h2>
<p style="margin:.55em 0">The server runs in this notebook on port 8010, against OpenAI this time. A fake browser speaks (Windows TTS),
reads the code off the simulated phone, and confirms the booking. The server's own logs and metrics are shown after.</p>

</div>

In [16]:
from websockets.asyncio.client import connect
from live import audio as tts   # notebooks' text-to-speech (dev only)
from app import logs
logs.setup('INFO')   # the server's own log lines appear below

live = settings.load(env={'APP__SERVER__PORT': '8010', 'APP__STORE__PATH': 'data/notebook08.db'})
store.init(live.path(live.store.path), seed_demo=True, reset=True)  # a fresh clinic: re-runs give the same result
ready, stop = asyncio.Event(), asyncio.Event()
srv = asyncio.create_task(server.run(live, ready=ready, stop=stop))
await ready.wait()
fri = store.next_weekday('Friday')
fri

2026-11-23 18:46:11,568 INFO [-] app.server: serving http://127.0.0.1:8010  agents=['front_desk', 'support']  toolsets={'appointments': 4, 'clinic_info': 2, 'identity': 3, 'orders': 2, 'registration': 1}


datetime.date(2026, 11, 27)

In [17]:
async def fake_browser(lines, agent='front_desk', limit=120):
    """Speaks line 0; reads the code off the phone when asked; says the last line once the caller is verified."""
    n, got, said, desk, sms, story = 2400, 0, set(), [], [], []
    async with connect(f'ws://127.0.0.1:8010/ws?agent={agent}', max_size=2**22) as ws:
        while json.loads(await ws.recv()).get('state') != 'ready':
            pass
        q = asyncio.Queue()
        async def mouth():
            while True:
                pcm = await asyncio.to_thread(tts.say, q.get_nowait()) if not q.empty() else None
                for i in range(0, len(pcm) if pcm is not None else n, n):
                    await ws.send(tts.to_bytes(pcm[i:i + n]) if pcm is not None else bytes(n * 2))
                    await ws.send(json.dumps({'played': got}))
                    await asyncio.sleep(0.1)
        talker = asyncio.create_task(mouth())
        q.put_nowait(lines[0])
        t0 = time.time()
        try:
            while time.time() - t0 < limit:
                try:
                    m = await asyncio.wait_for(ws.recv(), 40)
                except asyncio.TimeoutError:
                    if talker.done():
                        talker.result()  # the speaking task died: show why
                    raise
                if isinstance(m, bytes):
                    got += len(m); continue
                m = json.loads(m)
                if m['type'] == 'clear':
                    await ws.send(json.dumps({'cleared': 0, 'estimate': m['estimate']}))
                elif m['type'] == 'sms':
                    sms.append(m)
                elif m['type'] == 'desk':
                    desk.append(m)
                    if m['result'].get('status') == 'booked':
                        break
                elif m['type'] == 'error':
                    raise RuntimeError(m['message'])
                elif m['type'] == 'log' and m['event'] in ('user_said', 'model_said', 'tool_call', 'RESUME'):
                    story.append((m['event'], m['detail'][:140]))
                    verified = any(d['state']['patient'] for d in desk)
                    if m['event'] == 'model_said' and sms and 'code' not in said and 'code' in m['detail'].lower():
                        said.add('code'); q.put_nowait('The code is ' + ', '.join(re.sub(r'\D', '', sms[-1]['text'])) + '.')
                    elif m['event'] == 'model_said' and verified and 'yes' not in said and '?' in m['detail']:
                        said.add('yes'); q.put_nowait(lines[-1])
        finally:
            talker.cancel()
    return story, desk

In [18]:
story, desk = await fake_browser([
    f'Hi, I would like to book an appointment with Doctor Kapoor on {fri:%A} {fri.day} {fri:%B} at 11 AM '
    'for a checkup. My mobile is 9 8 4 5 0 1 2 3 6 7.',
    'Yes, please book it.'])
story

[('tool_call',
  'verify_patient({ "phone": "9845012367" }) -> {"status": "code_sent", "sent_to": "phone ending 2367", "next": "Ask for the 6-digit code that '),
 ('user_said',
  'Hi, I would like to book an appointment with Dr. Kapoor on Friday, November 27th at 11 a.m. for a checkup. My mobile is 9845012367.'),
 ('model_said',
  'I’ve sent a code to your mobile ending in 2367. Could you please provide me with the 6-digit code?'),
 ('tool_call',
  'confirm_code({ "code": "853981" }) -> {"status": "verified", "name": "Vikram Nair", "next": "Greet Vikram, then carry on with what they alre'),
 ('user_said', 'The code is 853981.'),
 ('model_said',
  'Hello Vikram, thank you for verifying. I’ll now proceed. \n\nSo, you’d like to book an appointment with Dr. Kapoor on Friday, November 27th at'),
 ('user_said', 'Yes, please book it.'),
 ('tool_call',
  'book_appointment({ "doctor": "Dr. Kapoor", "date": "2026-11-27", "time": "11:00 AM", "reason": "check-up" }) -> {"status": "booked", "appoin

In [19]:
[(d['skill'], d['result'].get('status')) for d in desk], desk[-1]['result']

([('verify_patient', 'code_sent'),
  ('confirm_code', 'verified'),
  ('book_appointment', 'booked')],
 {'status': 'booked',
  'appointment_id': 5,
  'doctor': 'Dr. Anil Kapoor',
  'date': 'Friday 27 November 2026',
  'time': '11:00 AM',
  'reason': 'check-up'})

In [20]:
await asyncio.sleep(1)  # let the finished conversation close
print('\n'.join(l for l in METRICS.render().splitlines() if not l.startswith('#') and ('tool_calls' in l or 'session' in l or 'resume' in l or 'stall' in l)))

resumes_total 1
sessions_active 1
sessions_rejected_total 1
sessions_total{agent="front_desk"} 2
stalls_total 1
tool_calls_total{skill="book_appointment",status="booked"} 2
tool_calls_total{skill="cancel_appointment",status="cancelled"} 1
tool_calls_total{skill="cancel_appointment",status="not_found"} 1
tool_calls_total{skill="choose_patient",status="verified"} 1
tool_calls_total{skill="confirm_code",status="already_verified"} 1
tool_calls_total{skill="confirm_code",status="choose_patient"} 1
tool_calls_total{skill="confirm_code",status="need_digits"} 1
tool_calls_total{skill="confirm_code",status="registered"} 1
tool_calls_total{skill="confirm_code",status="verified"} 1
tool_calls_total{skill="confirm_code",status="wrong_code"} 1
tool_calls_total{skill="my_appointments",status="needs_verification"} 1
tool_calls_total{skill="my_appointments",status="ok"} 1
tool_calls_total{skill="pharmacy_hours",status="ok"} 1
tool_calls_total{skill="start_registration",status="code_sent"} 1
tool_calls

In [21]:
stop.set()
await asyncio.wait_for(srv, 40)
srv.done()

True

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<span style="display:inline-block;margin-bottom:10px;padding:2px 12px;border-radius:999px;background:#6366f1;color:#fff;font:600 11px/20px system-ui,sans-serif;letter-spacing:.05em;text-transform:uppercase">Takeaways</span>
<ul style="margin:.4em 0;padding-left:1.4em">
  <li style="margin:.2em 0"><b>Scale by adding functions:</b> a typed, documented function in <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">app/skills/</code> is discovered, described,
      validated, guarded and timed out with no other edits; a new toolset is one name in config.</li>
  <li style="margin:.2em 0"><b>Tune without code:</b> prompts are files, every parameter is config, any value can come from the environment.</li>
  <li style="margin:.2em 0"><b>Production basics:</b> a slim dependency set (only <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">websockets</code>), limits, per-conversation logs, health and
      metrics endpoints, graceful drain, secrets from the environment, a container recipe.</li>
  <li style="margin:.2em 0"><b>Checked for free</b> with <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">python -m app.selfcheck</code>, including the server end to end against a fake OpenAI.</li>
</ul>
<p style="margin:.55em 0"><b>Next:</b> the course in <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">course/</code> rebuilds this backend step by step.</p>

</div>